In [2]:
# Verify which ldpc is being used
# RESTART KERNEL to use the local ldpc (v2.1.2) instead of pip version (v2.3.8)
import ldpc
print(f"ldpc version: {ldpc.__version__}")
print(f"ldpc location: {ldpc.__file__}")

ldpc version: 2.1.2
ldpc location: /root/Research/RithvikDecoder/Decoder/ldpc/src_python/ldpc/__init__.py


In [3]:
# NOTE: After installing local ldpc, restart the kernel to use it
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
sys.path.insert(0, os.path.join(project_root, 'ldpc', 'src_python'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /root/Research/RithvikDecoder/Decoder
First sys.path entry: /root/Research/RithvikDecoder/Decoder


---

In [4]:
import numpy as np
from scipy.sparse import csr_matrix, eye, hstack, save_npz, load_npz
import scipy.io

In [5]:
from utils.LDPC_encode import LDPCEncode
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER

In [6]:
from ldpc.bp_decoder import BpDecoder

---

In [7]:
H_mat_dat = scipy.io.loadmat('H.mat')
H = csr_matrix(H_mat_dat['H'])

In [8]:
decoder = BpDecoder(H, schedule="cluster")

In [9]:
n = 486 # length of message
n_frames = 10000
max_iter = 5

message = np.random.randint(0, 2, (n_frames, n))
print("Message shape:", message.shape)

Message shape: (10000, 486)


In [10]:
encoded_codeword = LDPCEncode(message)
print("Encoded codeword shape:", encoded_codeword.shape) 

tx_codeword = 1 - 2 * encoded_codeword 

Encoded codeword shape: (10000, 648)


In [11]:
m, _ = H.shape
arr = np.arange(m)
schedule = arr.reshape(6, -1)


In [12]:
snrs = [-3, -2, -1, 0, 1, 2, 3, 4]
bers = []

for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        for iter in range(max_iter):
            for cluster in schedule:
                llr = decoder.decode_cluster(cluster)
        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :n]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

BER at SNR -3 dB: 0.1581948559670782
BER at SNR -2 dB: 0.1298783950617284
BER at SNR -1 dB: 0.10084526748971194
BER at SNR 0 dB: 0.07017901234567901
BER at SNR 1 dB: 0.024186008230452676
BER at SNR 2 dB: 0.0006734567901234568
BER at SNR 3 dB: 4.11522633744856e-06
BER at SNR 4 dB: 0.0
